# release16_state_head_post_fullprobe_objective_screen_v1

Purpose: compare the latest release16 state-head full actor probe against the drawdown/upside checkpoint replay and the SOXLBacktest qlib reference, then decide the next model-selection/training objective without retaining an actor.

Baselines and role:

- canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`
- direct comparison: `release16_rulebase_reversal_drawdown_upside_checkpoint_metric_replay_v1` and latest full probe `release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1`
- reference baseline: `soxl_backtest_policy_reimplementation_v1` converted qlib daily path
- role: diagnose / checkpoint-objective handoff, not promotion

Acceptance criteria for any retained model remain: MaxDD `<20%`, walk-forward relative return `>=30pp` on `wf_2020`, `wf_2022`, `wf_2023_2024`, YTD return `>=` SOXLBacktest qlib reference and current-best alignment floor, plus replayable dashboard model evidence.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
for candidate_root in [ROOT, *ROOT.parents]:
    if (candidate_root / "src/soxl_quant").exists():
        ROOT = candidate_root
        break
else:
    raise RuntimeError("could not locate repo root containing src/soxl_quant")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.soxl_quant.app.state_conditioned_objective import (
    StateConditionedObjectiveConfig,
    score_state_conditioned_objective,
    summarize_state_conditioned_objective,
)

CARD = ROOT / "runtime/model_selection/card/release16_state_head_post_fullprobe_objective_screen_v1"
CARD.mkdir(parents=True, exist_ok=True)

FULL_PROFILE = "release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1"
FULL_CARD = ROOT / "runtime/model/card" / FULL_PROFILE
FULL_REWARD_CARD = ROOT / "runtime/reward/card" / FULL_PROFILE
CHECKPOINT_CARD = ROOT / "runtime/model_selection/card/release16_rulebase_reversal_drawdown_upside_checkpoint_metric_replay_v1"
ATTR_CARD = ROOT / "runtime/model_selection/card/release16_full_probe_failure_attribution_v1"
RETENTION_AUDIT = ROOT / "runtime/model_selection/card/retraining_retention_current_audit_v1/summary.json"
SOXLBACKTEST_DAILY = ROOT / "runtime/production/card/soxl_backtest_policy_reimplementation_v1/new_policy_daily.csv"
SAFE_END = pd.Timestamp("2026-06-09")
YTD_START = pd.Timestamp("2026-01-02")
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 20.0

with open(RETENTION_AUDIT) as fh:
    retention_summary = json.load(fh)
retention_gate = retention_summary["current_retention_gate"]
current_best_return_pct = float(retention_gate["current_best_return_pct"])
current_best_required_return_pct = float(retention_gate["required_best_return_pct"])
soxlbacktest_reference_return_pct = float(retention_gate["reference_return_pct"])

print("card", CARD.relative_to(ROOT))
print("current best", current_best_return_pct, "required", current_best_required_return_pct, "soxlbacktest", soxlbacktest_reference_return_pct)


card runtime/model_selection/card/release16_state_head_post_fullprobe_objective_screen_v1
current best 224.361539 required 213.143462 soxlbacktest 132.077978


In [2]:
def _normalise_daily(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    if "datetime" not in result.columns and "date" in result.columns:
        result = result.rename(columns={"date": "datetime"})
    if "datetime" not in result.columns:
        raise ValueError("daily frame missing datetime/date column")
    result["datetime"] = pd.to_datetime(result["datetime"])
    if "position" not in result.columns:
        for alias in ("target_position", "router_position", "base_position"):
            if alias in result.columns:
                result["position"] = result[alias]
                break
    if "position" not in result.columns:
        result["position"] = np.nan
    return result.sort_values("datetime").reset_index(drop=True)


def _pct_return(values: pd.Series) -> float:
    clean = pd.to_numeric(values, errors="coerce").dropna()
    if len(clean) < 2 or float(clean.iloc[0]) == 0.0:
        return 0.0
    return float((clean.iloc[-1] / clean.iloc[0] - 1.0) * 100.0)


def _max_drawdown(values: pd.Series) -> float:
    clean = pd.to_numeric(values, errors="coerce").dropna().astype(float)
    if len(clean) < 2:
        return 0.0
    peak = clean.cummax().replace(0.0, np.nan)
    dd = clean / peak - 1.0
    return abs(float(dd.min()) * 100.0)


def path_metrics(frame: pd.DataFrame, *, candidate: str, window: str, source: str) -> dict[str, object]:
    daily = _normalise_daily(frame)
    if daily.empty:
        return {
            "candidate": candidate,
            "window": window,
            "source": source,
            "rows": 0,
            "start": "",
            "end": "",
            "return_pct": 0.0,
            "buyhold_return_pct": 0.0,
            "relative_return_pp": float("-inf"),
            "max_dd_pct": float("inf"),
            "avg_position": 0.0,
            "trade_count": 0,
        }
    required_nav = {"strategy_nav", "buyhold_nav"}
    missing = required_nav - set(daily.columns)
    if missing:
        raise ValueError(f"{candidate}/{window} missing nav columns: {sorted(missing)}")
    ret = _pct_return(daily["strategy_nav"])
    bh = _pct_return(daily["buyhold_nav"])
    actions = daily.get("action", pd.Series(dtype=object)).astype(str).str.upper()
    return {
        "candidate": candidate,
        "window": window,
        "source": source,
        "rows": int(len(daily)),
        "start": str(daily["datetime"].iloc[0].date()),
        "end": str(daily["datetime"].iloc[-1].date()),
        "return_pct": ret,
        "buyhold_return_pct": bh,
        "relative_return_pp": ret - bh,
        "max_dd_pct": _max_drawdown(daily["strategy_nav"]),
        "avg_position": float(pd.to_numeric(daily["position"], errors="coerce").fillna(0.0).mean()),
        "trade_count": int(actions.isin(["BUY", "SELL"]).sum()) if len(actions) else 0,
    }


def slice_ytd(frame: pd.DataFrame) -> pd.DataFrame:
    daily = _normalise_daily(frame)
    return daily[(daily["datetime"] >= YTD_START) & (daily["datetime"] <= SAFE_END)].copy()


def read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


In [3]:
# Load current full actor probe daily paths and latest checkpoint official daily paths.
full_window_paths = {
    "wf_2020": FULL_CARD / f"{FULL_PROFILE}_wf_2020_soxl_daily.csv",
    "wf_2022": FULL_CARD / f"{FULL_PROFILE}_wf_2022_soxl_daily.csv",
    "wf_2023_2024": FULL_CARD / f"{FULL_PROFILE}_wf_2023_2024_soxl_daily.csv",
}
full_daily = read_csv(FULL_CARD / f"{FULL_PROFILE}_soxl_daily.csv")

candidate_paths: dict[str, dict[str, pd.DataFrame]] = {"full_actor_probe": {}}
for window, path in full_window_paths.items():
    candidate_paths["full_actor_probe"][window] = read_csv(path)

official_dir = CHECKPOINT_CARD / "official_daily_paths"
for path in sorted(official_dir.glob("*_official_daily_path.csv")):
    stem = path.stem.replace("_official_daily_path", "")
    window, raw_candidate = stem.split("__", 1)
    candidate = f"checkpoint_{raw_candidate}"
    candidate_paths.setdefault(candidate, {})[window] = read_csv(path)

metrics_rows = []
for candidate, by_window in candidate_paths.items():
    for window, frame in by_window.items():
        metrics_rows.append(path_metrics(frame, candidate=candidate, window=window, source="full_actor" if candidate == "full_actor_probe" else "checkpoint_official_daily"))

walkforward_metrics = pd.DataFrame(metrics_rows).sort_values(["candidate", "window"]).reset_index(drop=True)
walkforward_metrics["relative_gate"] = walkforward_metrics["relative_return_pp"] >= TARGET_RELATIVE_RETURN
walkforward_metrics["maxdd_gate"] = walkforward_metrics["max_dd_pct"] < TARGET_MAX_DD
walkforward_metrics["return_nonnegative_gate"] = walkforward_metrics["return_pct"] >= 0.0
walkforward_metrics["fold_gate"] = walkforward_metrics["relative_gate"] & walkforward_metrics["maxdd_gate"] & walkforward_metrics["return_nonnegative_gate"]
walkforward_metrics.to_csv(CARD / "walkforward_candidate_metrics.csv", index=False)

reported_full_probe = pd.read_csv(FULL_REWARD_CARD / "full_probe_metrics.csv")
reported_full_probe = reported_full_probe[reported_full_probe["window"].isin(["wf_2020", "wf_2022", "wf_2023_2024"])].copy()
recomputed_full_probe = walkforward_metrics[walkforward_metrics["candidate"] == "full_actor_probe"].copy()
metric_reconciliation = reported_full_probe.merge(
    recomputed_full_probe,
    on="window",
    how="outer",
    suffixes=("_reported_full_probe", "_daily_recomputed"),
)
for left, right, out in [
    ("return_pct_reported_full_probe", "return_pct_daily_recomputed", "return_delta_reported_minus_daily"),
    ("relative_return_pp_reported_full_probe", "relative_return_pp_daily_recomputed", "relative_delta_reported_minus_daily"),
    ("max_drawdown_pct", "max_dd_pct", "maxdd_delta_reported_minus_daily"),
]:
    metric_reconciliation[out] = pd.to_numeric(metric_reconciliation[left], errors="coerce") - pd.to_numeric(metric_reconciliation[right], errors="coerce")
metric_reconciliation["material_relative_mismatch"] = metric_reconciliation["relative_delta_reported_minus_daily"].abs() >= 5.0
metric_reconciliation.to_csv(CARD / "full_probe_metric_basis_reconciliation.csv", index=False)
metric_basis_summary = {
    "basis_used_for_candidate_screen": "official_daily_path_recomputed",
    "reported_full_probe_basis": "full_probe_summary_worker_metrics",
    "material_relative_mismatch_count": int(metric_reconciliation["material_relative_mismatch"].sum()),
    "material_relative_mismatch_windows": metric_reconciliation.loc[metric_reconciliation["material_relative_mismatch"], "window"].astype(str).tolist(),
    "gate_status_changed_by_recompute": False,
}
walkforward_metrics


,candidate,window,source,rows,start,end,return_pct,buyhold_return_pct,relative_return_pp,max_dd_pct,avg_position,trade_count,relative_gate,maxdd_gate,return_nonnegative_gate,fold_gate
0,checkpoint_bc_initialization,wf_2020,checkpoint_official_daily,252,2020-01-02,2020-12-30,57.666518,58.093571,-0.427053,19.326906,0.302925,30,False,True,True,False
1,checkpoint_bc_initialization,wf_2022,checkpoint_official_daily,250,2022-01-03,2022-12-29,-14.873092,-86.588072,71.714980,15.617043,0.081384,24,True,True,False,False
2,checkpoint_bc_initialization,wf_2023_2024,checkpoint_official_daily,251,2024-01-02,2024-12-30,2.635083,-0.356635,2.991718,13.624617,0.216822,30,False,True,True,False
3,checkpoint_ppo_retained_iter_0_bc_equivalent,wf_2020,checkpoint_official_daily,252,2020-01-02,2020-12-30,57.666518,58.093571,-0.427053,19.326906,0.302925,30,False,True,True,False
4,checkpoint_ppo_retained_iter_0_bc_equivalent,wf_2022,checkpoint_official_daily,250,2022-01-03,2022-12-29,-14.873092,-86.588072,71.714980,15.617043,0.081384,24,True,True,False,False
5,checkpoint_ppo_retained_iter_0_bc_equivalent,wf_2023_2024,checkpoint_official_daily,251,2024-01-02,2024-12-30,2.635083,-0.356635,2.991718,13.624617,0.216822,30,False,True,True,False
6,checkpoint_ppo_retained_iter_4,wf_2020,checkpoint_official_daily,252,2020-01-02,2020-12-30,44.378996,58.093571,-13.714574,25.585056,0.418305,31,False,False,True,False
7,checkpoint_ppo_retained_iter_4,wf_2022,checkpoint_official_daily,250,2022-01-03,2022-12-29,-18.398913,-86.588072,68.189159,19.877624,0.137760,24,True,True,False,False
8,checkpoint_ppo_retained_iter_4,wf_2023_2024,checkpoint_official_daily,251,2024-01-02,2024-12-30,-2.733554,-0.356635,-2.376920,13.164442,0.199528,28,False,True,False,False
9,checkpoint_ppo_selected_best,wf_2020,checkpoint_official_daily,252,2020-01-02,2020-12-30,44.378996,58.093571,-13.714574,25.585056,0.418305,31,False,False,True,False


In [4]:
# Score current candidates with the repo state-conditioned objective helper, using the user's MaxDD <20 gate.
metrics_by_candidate: dict[str, dict[str, dict[str, object]]] = {}
for row in walkforward_metrics.to_dict("records"):
    metrics_by_candidate.setdefault(str(row["candidate"]), {})[str(row["window"])] = {
        "relative_return": float(row["relative_return_pp"]),
        "max_dd": float(row["max_dd_pct"]),
        "avg_pos": float(row["avg_position"]),
    }

objective_config = StateConditionedObjectiveConfig(target_relative_return=TARGET_RELATIVE_RETURN, target_max_dd=TARGET_MAX_DD)
objective_scores = score_state_conditioned_objective(metrics_by_candidate, config=objective_config)
objective_summary = summarize_state_conditioned_objective(objective_scores)
objective_scores.to_csv(CARD / "walkforward_objective_scores.csv", index=False)
objective_summary


{'decision': 'no_objective_pass_candidate',
 'selected_candidate': 'checkpoint_bc_initialization',
 'selected_objective_loss': 57.43533457208876,
 'selected_window_gate': False,
 'selected_cluster_gate': False,
 'next_mechanism': 'train_state_conditioned_multi_objective',
 'required_terms': ['walkforward_relative_return_gap_penalty',
  'walkforward_drawdown_gap_penalty',
  'high_vol_rebound_participation_floor',
  'downtrend_drawdown_state_cap']}

In [5]:
# Gate matrix: full actor has fixed/YTD/WF evidence; checkpoint candidates only have fold-local WF evidence and cannot be retained by themselves.
ytd_full = slice_ytd(full_daily)
full_ytd_metrics = path_metrics(ytd_full, candidate="full_actor_probe", window="ytd_2026", source="full_actor_ytd")

soxl_ref = read_csv(SOXLBACKTEST_DAILY)
if "window" in soxl_ref.columns:
    soxl_ref_ytd = soxl_ref[soxl_ref["window"].astype(str) == "ytd_2026"].copy()
else:
    soxl_ref_ytd = slice_ytd(soxl_ref)
soxl_ref_ytd = slice_ytd(soxl_ref_ytd)
soxl_ref_metrics = path_metrics(soxl_ref_ytd, candidate="soxlbacktest_reference", window="ytd_2026", source="qlib_reference")

gate_rows = []

def add_gate(candidate: str, scope: str, gate: str, actual: float | str | bool | None, operator: str, limit: float | str | bool | None, passed: bool, note: str) -> None:
    gate_rows.append({
        "candidate": candidate,
        "scope": scope,
        "gate": gate,
        "actual": actual,
        "operator": operator,
        "limit": limit,
        "passed": bool(passed),
        "note": note,
    })

add_gate("full_actor_probe", "fixed_test_2020_2026", "fixed_test_max_drawdown", 25.329679974531977, "<", TARGET_MAX_DD, False, "from full_probe summary; violates user hard drawdown gate")
add_gate("full_actor_probe", "ytd_2026", "ytd_return_vs_soxlbacktest_reference", full_ytd_metrics["return_pct"], ">=", soxlbacktest_reference_return_pct, full_ytd_metrics["return_pct"] >= soxlbacktest_reference_return_pct, "must beat converted SOXLBacktest qlib reference")
add_gate("full_actor_probe", "ytd_2026", "ytd_return_vs_current_best_floor", full_ytd_metrics["return_pct"], ">=", current_best_required_return_pct, full_ytd_metrics["return_pct"] >= current_best_required_return_pct, "must align to current-best floor")
add_gate("full_actor_probe", "ytd_2026", "ytd_max_drawdown", full_ytd_metrics["max_dd_pct"], "<", TARGET_MAX_DD, full_ytd_metrics["max_dd_pct"] < TARGET_MAX_DD, "YTD drawdown sanity")

for row in walkforward_metrics.to_dict("records"):
    candidate = str(row["candidate"])
    window = str(row["window"])
    add_gate(candidate, window, "wf_relative_return", float(row["relative_return_pp"]), ">=", TARGET_RELATIVE_RETURN, bool(row["relative_gate"]), "walk-forward relative return floor")
    add_gate(candidate, window, "wf_max_drawdown", float(row["max_dd_pct"]), "<", TARGET_MAX_DD, bool(row["maxdd_gate"]), "walk-forward drawdown hard gate")
    add_gate(candidate, window, "wf_return_nonnegative", float(row["return_pct"]), ">=", 0.0, bool(row["return_nonnegative_gate"]), "fold return sanity")

for candidate in sorted(set(walkforward_metrics["candidate"]) - {"full_actor_probe"}):
    add_gate(candidate, "retention", "single_retained_actor_available", False, "==", True, False, "checkpoint paths are fold-local diagnostic paths, not one retained actor bundle")
    add_gate(candidate, "ytd_2026", "ytd_return_available", False, "==", True, False, "no single-actor YTD replay exists for checkpoint-only candidate")

add_gate("soxlbacktest_reference", "ytd_2026", "reference_return", soxl_ref_metrics["return_pct"], "reference", soxlbacktest_reference_return_pct, True, "converted qlib reference baseline; not runnable model inference")
add_gate("current_best_floor", "ytd_2026", "current_best_alignment_floor", current_best_required_return_pct, "floor", current_best_return_pct, True, "floor derived from current-best return")

gate_matrix = pd.DataFrame(gate_rows)
gate_matrix.to_csv(CARD / "candidate_gate_matrix.csv", index=False)
gate_matrix.groupby(["candidate", "passed"]).size().unstack(fill_value=0)


passed,False,True
candidate,,
checkpoint_bc_initialization,5,6
checkpoint_ppo_retained_iter_0_bc_equivalent,5,6
checkpoint_ppo_retained_iter_4,7,4
checkpoint_ppo_selected_best,7,4
current_best_floor,0,1
full_actor_probe,8,5
soxlbacktest_reference,0,1


In [6]:
# Compare the full actor against the best current checkpoint path by fold.
checkpoint_bc = walkforward_metrics[walkforward_metrics["candidate"] == "checkpoint_bc_initialization"].copy()
full_wf = walkforward_metrics[walkforward_metrics["candidate"] == "full_actor_probe"].copy()
full_vs_checkpoint = full_wf.merge(
    checkpoint_bc,
    on="window",
    suffixes=("_full", "_checkpoint_bc"),
)
for col in ["return_pct", "relative_return_pp", "max_dd_pct", "avg_position", "trade_count"]:
    full_vs_checkpoint[f"{col}_delta_full_minus_checkpoint"] = full_vs_checkpoint[f"{col}_full"] - full_vs_checkpoint[f"{col}_checkpoint_bc"]
full_vs_checkpoint.to_csv(CARD / "full_probe_vs_checkpoint_bc_gap.csv", index=False)
full_vs_checkpoint[[
    "window",
    "return_pct_full",
    "return_pct_checkpoint_bc",
    "return_pct_delta_full_minus_checkpoint",
    "relative_return_pp_full",
    "relative_return_pp_checkpoint_bc",
    "relative_return_pp_delta_full_minus_checkpoint",
    "max_dd_pct_full",
    "max_dd_pct_checkpoint_bc",
    "max_dd_pct_delta_full_minus_checkpoint",
    "avg_position_full",
    "avg_position_checkpoint_bc",
]]


,window,return_pct_full,return_pct_checkpoint_bc,return_pct_delta_full_minus_checkpoint,relative_return_pp_full,relative_return_pp_checkpoint_bc,relative_return_pp_delta_full_minus_checkpoint,max_dd_pct_full,max_dd_pct_checkpoint_bc,max_dd_pct_delta_full_minus_checkpoint,avg_position_full,avg_position_checkpoint_bc
0,wf_2020,35.374744,57.666518,-22.291774,-22.718827,-0.427053,-22.291774,19.260466,19.326906,-0.066440,0.184746,0.302925
1,wf_2022,-16.467158,-14.873092,-1.594066,70.120914,71.714980,-1.594066,18.149535,15.617043,2.532492,0.090647,0.081384
2,wf_2023_2024,-7.690929,2.635083,-10.326012,-7.334294,2.991718,-10.326012,20.914378,13.624617,7.289761,0.167797,0.216822


In [7]:
# YTD underparticipation vs SOXLBacktest reference: locate dates where the reference is materially invested and the full actor is nearly flat.
full_ytd_norm = _normalise_daily(ytd_full)[["datetime", "price", "strategy_nav", "buyhold_nav", "position", "action"]].rename(
    columns={
        "strategy_nav": "full_strategy_nav",
        "buyhold_nav": "full_buyhold_nav",
        "position": "full_position",
        "action": "full_action",
    }
)
ref_ytd_norm = _normalise_daily(soxl_ref_ytd)[["datetime", "price", "strategy_nav", "buyhold_nav", "position", "action"]].rename(
    columns={
        "strategy_nav": "ref_strategy_nav",
        "buyhold_nav": "ref_buyhold_nav",
        "position": "ref_position",
        "action": "ref_action",
    }
)
ytd_compare = ref_ytd_norm.merge(full_ytd_norm.drop(columns=["price"]), on="datetime", how="inner")
ytd_compare = ytd_compare.sort_values("datetime").reset_index(drop=True)
ytd_compare["position_gap_ref_minus_full"] = ytd_compare["ref_position"] - ytd_compare["full_position"]
ytd_compare["daily_price_return_next1"] = ytd_compare["price"].shift(-1) / ytd_compare["price"] - 1.0
ytd_compare["forward_price_return_5d"] = ytd_compare["price"].shift(-5) / ytd_compare["price"] - 1.0
ytd_compare["forward_price_return_10d"] = ytd_compare["price"].shift(-10) / ytd_compare["price"] - 1.0
ytd_compare["underparticipation_state"] = (ytd_compare["ref_position"] >= 0.35) & (ytd_compare["full_position"] <= 0.15)
ytd_compare["large_gap_state"] = ytd_compare["position_gap_ref_minus_full"] >= 0.25

ytd_under = ytd_compare[ytd_compare["underparticipation_state"]].copy()
ytd_under_top = ytd_under.sort_values(["position_gap_ref_minus_full", "forward_price_return_5d"], ascending=[False, False]).head(40)
ytd_compare.to_csv(CARD / "ytd_position_comparison_vs_soxlbacktest.csv", index=False)
ytd_under_top.to_csv(CARD / "ytd_underparticipation_days.csv", index=False)

def _safe_mean(series: pd.Series) -> float:
    clean = pd.to_numeric(series, errors="coerce").dropna()
    return float(clean.mean()) if len(clean) else 0.0

position_gap_summary = pd.DataFrame([
    {
        "metric": "all_ytd_rows",
        "value": int(len(ytd_compare)),
    },
    {
        "metric": "underparticipation_rows_ref_ge35_full_le15",
        "value": int(ytd_compare["underparticipation_state"].sum()),
    },
    {
        "metric": "large_gap_rows_ref_minus_full_ge25",
        "value": int(ytd_compare["large_gap_state"].sum()),
    },
    {
        "metric": "mean_ref_position",
        "value": _safe_mean(ytd_compare["ref_position"]),
    },
    {
        "metric": "mean_full_position",
        "value": _safe_mean(ytd_compare["full_position"]),
    },
    {
        "metric": "mean_position_gap_ref_minus_full",
        "value": _safe_mean(ytd_compare["position_gap_ref_minus_full"]),
    },
    {
        "metric": "underparticipation_forward_5d_price_return_mean_pct",
        "value": _safe_mean(ytd_under["forward_price_return_5d"]) * 100.0,
    },
    {
        "metric": "underparticipation_forward_10d_price_return_mean_pct",
        "value": _safe_mean(ytd_under["forward_price_return_10d"]) * 100.0,
    },
])
position_gap_summary.to_csv(CARD / "ytd_position_gap_summary.csv", index=False)
position_gap_summary


,metric,value
0,all_ytd_rows,109.000000
1,underparticipation_rows_ref_ge35_full_le15,20.000000
2,large_gap_rows_ref_minus_full_ge25,40.000000
3,mean_ref_position,0.162207
4,mean_full_position,0.085256
5,mean_position_gap_ref_minus_full,0.076951
6,underparticipation_forward_5d_price_return_mea...,19.067846
7,underparticipation_forward_10d_price_return_me...,29.001102


In [8]:
# Build a compact current decision table.
by_candidate = []
for candidate, group in walkforward_metrics.groupby("candidate"):
    by_candidate.append({
        "candidate": candidate,
        "folds_present": int(group["window"].nunique()),
        "fold_gate_count": int(group["fold_gate"].sum()),
        "all_fold_gate": bool(group["fold_gate"].all()),
        "min_relative_return_pp": float(group["relative_return_pp"].min()),
        "max_maxdd_pct": float(group["max_dd_pct"].max()),
        "mean_avg_position": float(group["avg_position"].mean()),
        "total_trades": int(group["trade_count"].sum()),
    })
candidate_summary = pd.DataFrame(by_candidate).sort_values(
    ["all_fold_gate", "fold_gate_count", "min_relative_return_pp", "max_maxdd_pct"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
objective_for_join = objective_scores.rename(columns={"min_relative_return": "objective_min_relative_return_pp", "max_max_dd": "objective_max_maxdd_pct"})
candidate_summary = candidate_summary.merge(
    objective_for_join[["candidate", "objective_loss", "relative_loss", "drawdown_loss", "participation_loss", "window_gate", "window_pass_count", "window_count", "worst_window"]],
    on="candidate",
    how="left",
)
candidate_summary.to_csv(CARD / "candidate_summary.csv", index=False)
candidate_summary


,candidate,folds_present,fold_gate_count,all_fold_gate,min_relative_return_pp,max_maxdd_pct,mean_avg_position,total_trades,objective_loss,relative_loss,drawdown_loss,participation_loss,window_gate,window_pass_count,window_count,worst_window
0,checkpoint_bc_initialization,3,0,False,-0.427053,19.326906,0.200377,84,57.435335,57.435335,0.000000,0.0,False,1,3,wf_2020
1,checkpoint_ppo_retained_iter_0_bc_equivalent,3,0,False,-0.427053,19.326906,0.200377,84,57.435335,57.435335,0.000000,0.0,False,1,3,wf_2020
2,checkpoint_ppo_retained_iter_4,3,0,False,-13.714574,25.585056,0.251864,83,131.942055,76.091494,55.850561,0.0,False,1,3,wf_2020
3,checkpoint_ppo_selected_best,3,0,False,-13.714574,25.585056,0.251864,83,131.942055,76.091494,55.850561,0.0,False,1,3,wf_2020
4,full_actor_probe,3,0,False,-22.718827,20.914378,0.147730,95,99.196903,90.053121,9.143782,0.0,False,1,3,wf_2020


In [9]:
selected_row = candidate_summary.iloc[0].to_dict() if not candidate_summary.empty else {}
full_actor_row = candidate_summary[candidate_summary["candidate"] == "full_actor_probe"].iloc[0].to_dict()
full_failed_gates = gate_matrix[(gate_matrix["candidate"] == "full_actor_probe") & (~gate_matrix["passed"])]["gate"].tolist()
checkpoint_fail_reason = "checkpoint candidates are fold-local diagnostic paths and lack one retained actor plus YTD replay"

under_summary = {row["metric"]: row["value"] for row in position_gap_summary.to_dict("records")}
summary = {
    "experiment": "release16_state_head_post_fullprobe_objective_screen_v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": "release16_state_head_post_fullprobe_objective_screen_no_actor_retention",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": [
        "release16_rulebase_reversal_drawdown_upside_checkpoint_metric_replay_v1",
        "release16_rulebase_reversal_drawdown_upside_state_head_actor_probe_v1",
    ],
    "reference_baseline": "soxl_backtest_policy_reimplementation_v1",
    "target_max_dd_pct": TARGET_MAX_DD,
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "current_best_return_pct": current_best_return_pct,
    "current_best_required_return_pct": current_best_required_return_pct,
    "soxlbacktest_reference_return_pct": soxlbacktest_reference_return_pct,
    "full_actor_ytd": full_ytd_metrics,
    "soxlbacktest_ytd_reference": soxl_ref_metrics,
    "selected_diagnostic_candidate": selected_row.get("candidate", ""),
    "selected_diagnostic_candidate_summary": selected_row,
    "full_actor_summary": full_actor_row,
    "full_actor_failed_gates": full_failed_gates,
    "metric_basis_reconciliation": metric_basis_summary,
    "objective_summary": objective_summary,
    "ytd_underparticipation_summary": under_summary,
    "training_allowed_from_this_card": False,
    "next_training_allowed_after_new_contract": True,
    "next_mechanism": "checkpoint_distillation_with_ytd_participation_floor_and_ppo_drift_stop",
    "required_next_terms": [
        "official_daily_path_checkpoint_objective_loss",
        "ytd_soxlbacktest_position_alignment_loss",
        "walkforward_relative_return_gap_penalty",
        "walkforward_drawdown_gap_penalty",
        "ppo_vs_bc_drawdown_drift_stop",
        "retain_actor_only_after_full_fixed_wf_ytd_dashboard_replay_gates",
    ],
    "conclusion": [
        "The full state-head actor remains non-retainable: it fails fixed MaxDD, YTD return vs SOXLBacktest/current-best, and multiple walk-forward gates.",
        "The latest drawdown/upside checkpoint BC path keeps all three walk-forward MaxDD values below 20%, but it still fails relative-return gates on wf_2020 and wf_2023_2024 and is not a single retained actor.",
        "Candidate comparison uses official daily-path recomputed metrics; reported full-probe worker metrics are preserved in full_probe_metric_basis_reconciliation.csv and do not change the gate decision.",
        "PPO/full-actor training drifted away from the checkpoint path: relative return and YTD participation deteriorated rather than becoming a retainable model.",
        "Next work should not rerun this profile unchanged; create a new contract that distills the checkpoint path, adds a YTD SOXLBacktest position-alignment floor, and enforces PPO-vs-BC drawdown drift stops before any actor retention review.",
    ],
    "sidecars": [
        "walkforward_candidate_metrics.csv",
        "walkforward_objective_scores.csv",
        "candidate_gate_matrix.csv",
        "full_probe_metric_basis_reconciliation.csv",
        "full_probe_vs_checkpoint_bc_gap.csv",
        "ytd_position_comparison_vs_soxlbacktest.csv",
        "ytd_position_gap_summary.csv",
        "ytd_underparticipation_days.csv",
        "candidate_summary.csv",
        "selection.json",
        "summary.json",
        "conclusion.md",
    ],
}
selection = {
    key: summary[key]
    for key in [
        "experiment",
        "created_at",
        "decision",
        "promotion_allowed",
        "actor_retention_allowed",
        "selected_diagnostic_candidate",
        "selected_diagnostic_candidate_summary",
        "full_actor_failed_gates",
        "next_mechanism",
        "required_next_terms",
        "training_allowed_from_this_card",
        "next_training_allowed_after_new_contract",
    ]
}
selection["checkpoint_candidate_blocker"] = checkpoint_fail_reason
selection["hard_blocks"] = [
    "Do not retain full_actor_probe; generated actors were already removed and gates fail.",
    "Do not promote checkpoint-only candidates; they are fold-local diagnostics without single-actor YTD replay.",
    "Do not relax SOXLBacktest/current-best YTD return floors or the MaxDD <20 gate.",
]

for name, obj in [("summary.json", summary), ("selection.json", selection)]:
    with open(CARD / name, "w") as fh:
        json.dump(obj, fh, indent=2, sort_keys=True)

conclusion = "\n".join([
    "# release16 state-head post-fullprobe objective screen conclusion",
    "",
    f"Decision: `{summary['decision']}`.",
    "",
    "The latest full state-head actor is not retainable. It still fails the fixed-test MaxDD gate, YTD return alignment against both SOXLBacktest and current-best floor, and walk-forward stability gates.",
    "",
    f"The best diagnostic candidate by current fold evidence is `{summary['selected_diagnostic_candidate']}`. It is useful only as a checkpoint/objective teacher because checkpoint paths are fold-local and have no single retained actor or YTD replay.",
    "",
    f"YTD underparticipation rows where SOXLBacktest position >=35% and full actor position <=15%: {int(under_summary.get('underparticipation_rows_ref_ge35_full_le15', 0))}. Mean full position is {float(under_summary.get('mean_full_position', 0.0)):.3f} vs SOXLBacktest {float(under_summary.get('mean_ref_position', 0.0)):.3f}.",
    "",
    "Candidate-screen metrics are recomputed from official daily paths for comparability; see full_probe_metric_basis_reconciliation.csv for the reported full-probe worker-metric deltas.",
    "",
    "Next action: build a new checkpoint-distillation / target contract with YTD SOXLBacktest position-alignment loss and a PPO-vs-BC drawdown drift stop, then run a constrained replay before any retained actor training.",
    "",
])
(CARD / "conclusion.md").write_text(conclusion)
print(json.dumps(selection, indent=2)[:4000])


{
  "experiment": "release16_state_head_post_fullprobe_objective_screen_v1",
  "created_at": "2026-06-14T18:49:21.424615+00:00",
  "decision": "release16_state_head_post_fullprobe_objective_screen_no_actor_retention",
  "promotion_allowed": false,
  "actor_retention_allowed": false,
  "selected_diagnostic_candidate": "checkpoint_bc_initialization",
  "selected_diagnostic_candidate_summary": {
    "candidate": "checkpoint_bc_initialization",
    "folds_present": 3,
    "fold_gate_count": 0,
    "all_fold_gate": false,
    "min_relative_return_pp": -0.4270525146105939,
    "max_maxdd_pct": 19.326906429203994,
    "mean_avg_position": 0.20037689289028449,
    "total_trades": 84,
    "objective_loss": 57.43533457208876,
    "relative_loss": 57.43533457208876,
    "drawdown_loss": 0.0,
    "participation_loss": 0.0,
    "window_gate": false,
    "window_pass_count": 1,
    "window_count": 3,
    "worst_window": "wf_2020"
  },
  "full_actor_failed_gates": [
    "fixed_test_max_drawdown",
   

## Conclusion

This notebook writes `summary.json`, `selection.json`, and CSV sidecars. The conclusion is intentionally not a promotion: the current full actor is non-retainable, and checkpoint-only paths are diagnostic teachers until a single actor passes fixed split, walk-forward, YTD reference, model replay, and dashboard runnable-source gates.
